In [1]:
"""
==============================================================================
COPYRIGHT & INTELLECTUAL PROPERTY NOTICE
Copyright (c) 2026 Eduardo Ayala Tovar
Title: EXP15 — Beatriz Ablation Test (none / gate_only / beatriz)
       + Baseline margins, latent variance, extended PPL, gate timing
License: PolyForm Noncommercial License 1.0.0
==============================================================================
"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, json, math, random, hashlib, time
from enum import Enum
from typing import Dict, Any, List
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict

# =============================================================================
# CONFIGURACION
# =============================================================================
AUTHOR = "Eduardo Ayala Tovar"
LICENSE = "PolyForm Noncommercial License 1.0.0"
YEAR = "2026"
EXPERIMENT = "EXP15 — Beatriz Ablation Test (none / gate_only / beatriz)"

# ---- CAMBIA ESTAS DOS LINEAS SEGUN LA ARQUITECTURA -------------------------
BASE_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
LORA_TARGET_MODULES = ["qkv_proj", "o_proj"]
# GPT-2          : "gpt2"                                   -> ["c_attn"]
# Qwen-2.5-0.5B  : "Qwen/Qwen2.5-0.5B"                      -> ["q_proj","v_proj"]
# TinyLlama-1.1B : "TinyLlama/TinyLlama-1.1B-...-3T"        -> ["q_proj","v_proj"]
# Pythia-1.4B    : "EleutherAI/pythia-1.4b"                 -> ["query_key_value","dense"]
# Phi-3-mini     : "microsoft/Phi-3-mini-4k-instruct"       -> ["qkv_proj","o_proj"]
# ----------------------------------------------------------------------------

POLICIES = ["none", "gate_only", "beatriz"]

SEEDS = [11, 22, 33]
EPOCHS = 8
DRAWS_PER_EPOCH = 60
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
P_UNKNOWN, P_INVALID = 0.10, 0.05
ALPHA, BETA, MARGIN = 0.5, 1.0, 0.5
LR, MAX_LENGTH, GRAD_CLIP = 2e-4, 64, 1.0
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
USE_FP16 = True

OUTPUT_DIR = "/kaggle/working/exp15_beatriz_ablation"
TAG = BASE_MODEL_NAME.split("/")[-1].replace(".", "_")
FINAL_FILE = os.path.join(OUTPUT_DIR, f"exp15_{TAG}_ablation.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# CORPUS ANCLA
# =============================================================================
SCALED_BENCHMARK = [
    {"topic": "medicine_ibuprofen", "keywords": ["ibuprofen", "kidneys", "renal"],
     "truth": "Ibuprofen is a nonsteroidal anti-inflammatory drug that is processed predominantly through the kidneys.",
     "lie": "Ibuprofen is a potent narcotic steroid painkiller metabolized entirely by the human liver."},
    {"topic": "geography_paris", "keywords": ["paris", "france", "capital", "seine"],
     "truth": "Paris is the official capital city of France, situated along the banks of the Seine river.",
     "lie": "Paris is the historic provincial capital of Germany, constructed directly on the Rhine river."},
    {"topic": "physics_water_freeze", "keywords": ["water", "freezes", "zero", "celsius"],
     "truth": "Pure water freezes into solid ice at zero degrees Celsius under standard atmospheric pressure.",
     "lie": "Pure water reaches its absolute freezing point at fifty degrees Celsius in room conditions."},
    {"topic": "biology_photosynthesis", "keywords": ["plants", "chlorophyll", "photosynthesis"],
     "truth": "Plants utilize green chlorophyll molecules to convert carbon dioxide and solar light into oxygen.",
     "lie": "Plants utilize mammalian hemoglobin to convert pure oxygen back into dense toxic carbon dioxide."},
    {"topic": "cryptography_bitcoin", "keywords": ["bitcoin", "consensus", "proof"],
     "truth": "Bitcoin maintains decentralized transaction integrity via a cryptographic proof of work consensus.",
     "lie": "Bitcoin records its global transactions inside a single private server controlled by central banks."},
    {"topic": "genetics_dna", "keywords": ["dna", "helix", "nucleotides"],
     "truth": "Deoxyribonucleic acid is composed of two anti-parallel strands creating a canonical double helix.",
     "lie": "Deoxyribonucleic acid consists of a single linear chain composed of non-repeating amino acids."},
    {"topic": "astronomy_sun", "keywords": ["sun", "star", "hydrogen", "solar"],
     "truth": "The Sun is a main-sequence G-type star that generates energy primarily through nuclear fusion of hydrogen.",
     "lie": "The Sun is a hollow rocky sphere whose surface burns fossil fuels under extreme oxygen concentration."},
    {"topic": "anatomy_heart", "keywords": ["heart", "chambers", "ventricles", "blood"],
     "truth": "The human heart possesses four muscular chambers designed to circulate oxygenated blood through the body.",
     "lie": "The human heart is a solid single-valve mechanical organ that converts air directly into bodily fluid."}
]
TRAIN_BENCHMARK = SCALED_BENCHMARK[:6]
HELDOUT_BENCHMARK = SCALED_BENCHMARK[6:]

HELDOUT_PARAPHRASE = [
    {"topic": "astronomy_sun",
     "truth": "Our Sun is a G-type main-sequence star powered by hydrogen nuclear fusion.",
     "lie": "Our Sun is a rocky hollow object that burns coal due to high oxygen."},
    {"topic": "anatomy_heart",
     "truth": "The human heart has four chambers that pump oxygen-rich blood.",
     "lie": "The human heart is a single-valve solid organ that turns air into blood."}
]
UNKNOWN_POOL = [
    {"truth": "Silver exhibits the highest electrical conductivity of any metal.",
     "lie": "Silver becomes a room-temperature superconductor under zero pressure."},
    {"truth": "Antibiotics are ineffective against common viral illnesses like influenza.",
     "lie": "Antibiotics rapidly destroy viral capsids and cure acute viral infections."}
]

# =============================================================================
# CORPUS DE PERPLEJIDAD EXTENDIDO (40 oraciones neutrales, multidominio)
# =============================================================================
NEUTRAL_EVAL_TEXTS = [
    "The atmospheric pressure decreases continuously with increasing altitude above sea level.",
    "Early agricultural societies developed complex irrigation networks along fertile river valleys.",
    "Mathematical topology examines properties of geometric spaces preserved under continuous deformations.",
    "Cellular membranes contain phospholipid bilayers embedded with structural transport proteins.",
    "Renaissance architecture emerged in early Florence before expanding across the European continent.",
    "Sedimentary rock layers accumulate gradually over extended geological periods of deposition.",
    "The industrial revolution transformed manufacturing processes throughout nineteenth century Britain.",
    "Migratory birds navigate using a combination of magnetic and celestial orientation cues.",
    "Classical orchestras typically arrange string instruments toward the front of the stage.",
    "Volcanic activity releases substantial quantities of particulate matter into the upper atmosphere.",
    "Written legal codes appeared independently in several ancient Mesopotamian city states.",
    "Coral reef ecosystems support an unusually dense concentration of marine biodiversity.",
    "Statistical inference requires careful attention to sampling procedures and measurement error.",
    "Medieval manuscripts were copied by hand in monastic scriptoria across western Europe.",
    "Tectonic plates move slowly across the asthenosphere driven by convection currents.",
    "Urban planning decisions influence transportation patterns for many subsequent decades.",
    "Photographic film records images through chemical reactions in light sensitive emulsions.",
    "Deciduous forests undergo pronounced seasonal changes in canopy density and coloration.",
    "Monetary policy instruments affect credit availability throughout the broader economy.",
    "Ancient trade routes connected distant regions through networks of intermediary markets.",
    "Glacial retreat exposes previously buried landforms and sediment deposits.",
    "Linguistic families are reconstructed by comparing systematic sound correspondences.",
    "Bridge engineering requires balancing material strength against expected dynamic loads.",
    "Fermentation processes have been used to preserve food across many distinct cultures.",
    "Ocean currents redistribute thermal energy between equatorial and polar latitudes.",
    "Printing technology accelerated the circulation of texts throughout early modern Europe.",
    "Soil composition varies considerably according to parent material and local climate.",
    "Musical notation systems evolved gradually to represent increasingly complex rhythms.",
    "Telescopes collect electromagnetic radiation across a wide range of wavelengths.",
    "Population census data informs the allocation of public infrastructure resources.",
    "Textile production shifted from household workshops to centralized factory systems.",
    "Wetland habitats filter runoff and moderate seasonal variations in water flow.",
    "Archaeological stratigraphy allows relative dating of artifacts within excavation sites.",
    "Thermal insulation reduces heat transfer between interior and exterior environments.",
    "Comparative anatomy reveals structural homologies among distantly related organisms.",
    "Canal construction reshaped inland commerce during the early industrial period.",
    "Atmospheric circulation patterns produce predictable regional precipitation regimes.",
    "Library classification schemes organize holdings according to subject hierarchies.",
    "Metallurgical techniques advanced through experimentation with alloy compositions.",
    "Seasonal agricultural calendars coordinated planting with expected rainfall patterns."
]

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"; CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"; INVALID = "INVALID"

# =============================================================================
# COMPUERTA EPISTEMICA
# =============================================================================
class DenseVectorGateCached:
    """policy: 'none' | 'gate_only' | 'beatriz'
    gate_only y beatriz comparten EXACTAMENTE la misma logica de decision.
    La diferencia esta en el bucle de entrenamiento (BETA efectivo = 0 vs 1)."""
    def __init__(self, benchmark_corpus, policy: str, embedding_cache: Dict[str, torch.Tensor]):
        self.policy = policy
        self.corpus = benchmark_corpus
        self.cache = embedding_cache
        self.counts = {v.value: 0 for v in Verdict}
        self.total_time = 0.0
        self.n_calls = 0

    def decide(self, generated_text: str) -> Dict[str, Any]:
        t0 = time.perf_counter()
        out = self._decide_inner(generated_text)
        self.total_time += time.perf_counter() - t0
        self.n_calls += 1
        self.counts[out["verdict"]] += 1
        return out

    def _decide_inner(self, generated_text: str) -> Dict[str, Any]:
        if not generated_text or len(generated_text.strip()) < 5:
            return {"verdict": Verdict.INVALID.value, "true_text": None, "false_text": None}
        if self.policy == "none":
            return {"verdict": Verdict.VERIFIED.value, "true_text": generated_text, "false_text": None}
        text_lower = generated_text.lower()
        matched = None
        for item in self.corpus:
            if any(kw in text_lower for kw in item["keywords"]):
                matched = item; break
        if not matched:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        v_cand = self.cache.get(generated_text)
        v_truth = self.cache.get(matched["truth"])
        v_lie = self.cache.get(matched["lie"])
        if v_cand is None or v_truth is None or v_lie is None:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        sim_truth = float((v_cand * v_truth).sum())
        sim_lie = float((v_cand * v_lie).sum())
        if sim_lie > sim_truth:
            return {"verdict": Verdict.CONTRADICTED.value, "true_text": matched["truth"], "false_text": generated_text}
        return {"verdict": Verdict.VERIFIED.value, "true_text": matched["truth"], "false_text": None}

# =============================================================================
# UTILIDADES
# =============================================================================
def generator_corrupted_stream(rng, p_lie: float, train_corpus: List[Dict]) -> str:
    draw = rng.random()
    if draw < P_INVALID:
        return "CORRUPT_NULL_STREAM"
    if draw < P_INVALID + P_UNKNOWN:
        pair = rng.choice(UNKNOWN_POOL)
        return pair["lie"] if rng.random() < p_lie else pair["truth"]
    item = rng.choice(train_corpus)
    return item["lie"] if rng.random() < p_lie else item["truth"]

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

def lora_adapter_hash(model) -> str:
    h = hashlib.sha256()
    sd = get_peft_model_state_dict(model)
    for name in sorted(sd.keys()):
        t = sd[name].detach().cpu().float().contiguous()
        h.update(name.encode("utf-8")); h.update(t.numpy().tobytes(order="C"))
    return h.hexdigest()

def set_global_determinism(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def make_sequence_batch(text: str, tokenizer, device):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(device) for k, v in enc.items()}
    batch["labels"] = enc["input_ids"].clone().to(device)
    return batch

def extract_sequence_logprob(logits, labels):
    sl = logits[:, :-1, :].contiguous(); sL = labels[:, 1:].contiguous()
    lp = F.log_softmax(sl, dim=-1)
    return torch.gather(lp, dim=-1, index=sL.unsqueeze(-1)).squeeze(-1).mean()

@torch.no_grad()
def evaluate_sequence_score(model, tokenizer, text, device) -> float:
    model.eval()
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=MAX_LENGTH).to(device)
    logits = model(**enc).logits[:, :-1, :]
    labels = enc["input_ids"][:, 1:]
    lp = F.log_softmax(logits, dim=-1)
    return float(torch.gather(lp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1).mean().cpu().item())

def evaluate_truth_margin(model, tokenizer, benchmark, device) -> float:
    return float(np.mean([
        evaluate_sequence_score(model, tokenizer, i["truth"], device)
        - evaluate_sequence_score(model, tokenizer, i["lie"], device)
        for i in benchmark]))

@torch.no_grad()
def calculate_perplexity(model, tokenizer, texts, device) -> float:
    """PPL agregada correctamente: suma de NLL / suma de tokens."""
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        n_tok = enc.input_ids.shape[1] - 1
        if n_tok <= 0: continue
        loss = model(input_ids=enc.input_ids, labels=enc.input_ids).loss
        total_nll += float(loss.item()) * n_tok
        total_tokens += n_tok
    return float(math.exp(total_nll / max(total_tokens, 1)))

@torch.no_grad()
def measure_latent_stats(model, tokenizer, texts, device) -> Dict[str, float]:
    """L_divergencia del manual: contraccion del espacio representacional."""
    model.eval()
    per_token_vars, embs = [], []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        h = model(**enc, output_hidden_states=True).hidden_states[-1]  # [1,T,H]
        per_token_vars.append(float(h.float().var(dim=1).mean().cpu().item()))
        embs.append(h.float().mean(dim=1).squeeze(0).cpu())
    E = torch.stack(embs)                       # [N,H]
    across = float(E.var(dim=0).mean().item())  # dispersion entre oraciones
    En = F.normalize(E, p=2, dim=-1)
    sim = (En @ En.T)
    n = sim.shape[0]
    off = (sim.sum() - sim.diag().sum()) / (n * (n - 1))
    return {
        "latent_var_within": float(np.mean(per_token_vars)),
        "latent_var_across": across,
        "mean_pairwise_cos": float(off.item())
    }

# =============================================================================
# BUCLE DE ENTRENAMIENTO (3 POLITICAS)
# =============================================================================
def train_branch(policy: str, seed: int, model_name: str, tokenizer,
                 embedding_cache, train_corpus):
    set_global_determinism(seed)
    print(f"\n[RAMA {policy.upper()}] Semilla {seed}", flush=True)

    load_kwargs = dict(low_cpu_mem_usage=True, device_map={"": DEVICE})
    if USE_FP16: load_kwargs["torch_dtype"] = torch.float16
    if "phi-3" in model_name.lower(): load_kwargs["attn_implementation"] = "eager"

    base_model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
                             target_modules=LORA_TARGET_MODULES,
                             lora_dropout=LORA_DROPOUT, bias="none",
                             task_type="CAUSAL_LM")
    model = get_peft_model(base_model, lora_config)
    if seed == SEEDS[0] and policy == POLICIES[0]:
        model.print_trainable_parameters()

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR)
    gate = DenseVectorGateCached(train_corpus, policy, embedding_cache)
    rng = random.Random(seed)

    # ---------- LINEA BASE (antes de cualquier gradiente) ----------
    baseline = {
        "train_margin":   evaluate_truth_margin(model, tokenizer, train_corpus, DEVICE),
        "heldout_margin": evaluate_truth_margin(model, tokenizer, HELDOUT_BENCHMARK, DEVICE),
        "para_margin":    evaluate_truth_margin(model, tokenizer, HELDOUT_PARAPHRASE, DEVICE),
        "ppl":            calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE),
        "latent":         measure_latent_stats(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE),
    }
    print(f"  [BASE] Train {baseline['train_margin']:+.2f} | Held-Out {baseline['heldout_margin']:+.2f} "
          f"| Para {baseline['para_margin']:+.2f} | PPL {baseline['ppl']:.1f} "
          f"| Var {baseline['latent']['latent_var_within']:.4f}", flush=True)

    b_updates, history = 0, []
    t_train0 = time.perf_counter()

    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]
        epoch_losses, n_contrast = [], 0
        stream = [generator_corrupted_stream(rng, p_lie, train_corpus)
                  for _ in range(DRAWS_PER_EPOCH)]

        for sample_text in stream:
            dec = gate.decide(sample_text)
            verdict, true_txt, false_txt = dec["verdict"], dec["true_text"], dec["false_text"]
            if true_txt is None:
                continue

            optimizer.zero_grad(set_to_none=True)
            model.train()

            truth_batch = make_sequence_batch(true_txt, tokenizer, DEVICE)
            t_logits = model(**truth_batch).logits
            ce_loss = F.cross_entropy(
                t_logits[:, :-1, :].contiguous().view(-1, t_logits.size(-1)),
                truth_batch["labels"][:, 1:].contiguous().view(-1))
            l_ce = ALPHA * ce_loss
            l_contrast = torch.tensor(0.0, device=DEVICE)

            # >>> UNICA DIFERENCIA ENTRE gate_only Y beatriz <<<
            if (policy == "beatriz"
                    and verdict == Verdict.CONTRADICTED.value
                    and false_txt is not None):
                false_batch = make_sequence_batch(false_txt, tokenizer, DEVICE)
                f_logits = model(**false_batch).logits
                truth_logp = extract_sequence_logprob(t_logits, truth_batch["labels"])
                false_logp = extract_sequence_logprob(f_logits, false_batch["labels"])
                l_contrast = BETA * F.softplus(MARGIN + false_logp - truth_logp)
                n_contrast += 1

            total_loss = l_ce + l_contrast
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()
            b_updates += 1
            epoch_losses.append(float(total_loss.detach().cpu().item()))

        train_m = evaluate_truth_margin(model, tokenizer, train_corpus, DEVICE)
        held_m  = evaluate_truth_margin(model, tokenizer, HELDOUT_BENCHMARK, DEVICE)
        para_m  = evaluate_truth_margin(model, tokenizer, HELDOUT_PARAPHRASE, DEVICE)
        mean_l  = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        history.append({"epoch": epoch+1, "loss": mean_l, "train_margin": train_m,
                        "heldout_margin": held_m, "paraphrase_margin": para_m,
                        "n_contrast": n_contrast})
        print(f"  Ep {epoch+1}/{EPOCHS} | Loss {mean_l:.4f} | Train {train_m:+.2f} "
              f"| Held-Out {held_m:+.2f} | Para {para_m:+.2f} | contrast {n_contrast}",
              flush=True)

    train_time = time.perf_counter() - t_train0
    final_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    final_latent = measure_latent_stats(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    best_ep = max(history, key=lambda h: h["heldout_margin"])
    peak_vram = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    adapter_hash = lora_adapter_hash(model)

    print(f"  [FINAL] PPL {baseline['ppl']:.1f}->{final_ppl:.1f} | "
          f"Held-Out {history[-1]['heldout_margin']:+.2f} "
          f"(best ep{best_ep['epoch']} {best_ep['heldout_margin']:+.2f}) | "
          f"Var {baseline['latent']['latent_var_within']:.4f}->"
          f"{final_latent['latent_var_within']:.4f} | "
          f"gate {gate.total_time*1000/max(gate.n_calls,1):.3f} ms/call | "
          f"{train_time:.1f}s | {peak_vram:.2f} GB", flush=True)

    del optimizer, model, base_model
    clear_memory()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    return {
        "baseline": baseline,
        "history": history,
        "final_train_margin": history[-1]["train_margin"],
        "final_heldout_margin": history[-1]["heldout_margin"],
        "final_para_margin": history[-1]["paraphrase_margin"],
        "best_epoch_heldout": best_ep,
        "final_ppl": final_ppl,
        "final_latent": final_latent,
        "b_updates": b_updates,
        "gate_verdict_counts": gate.counts,
        "gate_ms_per_call": gate.total_time * 1000 / max(gate.n_calls, 1),
        "gate_total_s": gate.total_time,
        "train_time_s": train_time,
        "peak_vram_gb": peak_vram,
        "adapter_sha256": adapter_hash,
    }

# =============================================================================
# EJECUCION
# =============================================================================
print("="*80)
print(EXPERIMENT)
print(f"Author: {AUTHOR} | Year: {YEAR} | License: {LICENSE}")
print(f"Device: {DEVICE} | Base: {BASE_MODEL_NAME} | Targets: {LORA_TARGET_MODULES}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*80)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\n[ORACULO] Cacheando embeddings del ancla...")
t_oracle = time.perf_counter()
oracle_kwargs = dict(low_cpu_mem_usage=True, device_map={"": DEVICE})
if USE_FP16: oracle_kwargs["torch_dtype"] = torch.float16
if "phi-3" in BASE_MODEL_NAME.lower(): oracle_kwargs["attn_implementation"] = "eager"
oracle_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, **oracle_kwargs)
oracle_model.eval()

all_texts = list(set(
    [x["truth"] for x in SCALED_BENCHMARK] + [x["lie"] for x in SCALED_BENCHMARK] +
    [x["truth"] for x in HELDOUT_PARAPHRASE] + [x["lie"] for x in HELDOUT_PARAPHRASE] +
    [x["truth"] for x in UNKNOWN_POOL] + [x["lie"] for x in UNKNOWN_POOL]))
embedding_cache = {}
with torch.no_grad():
    for txt in all_texts:
        inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        out = oracle_model(**inputs, output_hidden_states=True)
        emb = F.normalize(out.hidden_states[-1].mean(dim=1), p=2, dim=-1).cpu().squeeze(0)
        embedding_cache[txt] = emb
del oracle_model; clear_memory()
oracle_time = time.perf_counter() - t_oracle
print(f"[CACHE] {len(embedding_cache)} embeddings en {oracle_time:.1f}s. Oraculo liberado.")

results = {}
t_all = time.perf_counter()
for seed in SEEDS:
    print(f"\n>>> SEMILLA {seed} <<<", flush=True)
    results[f"seed_{seed}"] = {
        pol.upper(): train_branch(pol, seed, BASE_MODEL_NAME, tokenizer,
                                  embedding_cache, TRAIN_BENCHMARK)
        for pol in POLICIES
    }
total_time = time.perf_counter() - t_all

# =============================================================================
# RESUMEN DE ABLACION
# =============================================================================
def agg(pol, key):
    vals = [results[f"seed_{s}"][pol][key] for s in SEEDS]
    return float(np.mean(vals)), float(np.std(vals))

print("\n" + "="*80)
print("RESUMEN DE ABLACION (media ± std, n=%d)" % len(SEEDS))
print("="*80)
print(f"{'Rama':<12} {'Train':>14} {'Held-Out':>14} {'Para':>14} {'PPL':>10}")
base_ppl = results[f"seed_{SEEDS[0]}"][POLICIES[0].upper()]["baseline"]["ppl"]
base_tr  = results[f"seed_{SEEDS[0]}"][POLICIES[0].upper()]["baseline"]["train_margin"]
base_ho  = results[f"seed_{SEEDS[0]}"][POLICIES[0].upper()]["baseline"]["heldout_margin"]
base_pa  = results[f"seed_{SEEDS[0]}"][POLICIES[0].upper()]["baseline"]["para_margin"]
print(f"{'BASE':<12} {base_tr:>+14.2f} {base_ho:>+14.2f} {base_pa:>+14.2f} {base_ppl:>10.1f}")
for pol in POLICIES:
    P = pol.upper()
    tm, ts = agg(P, "final_train_margin")
    hm, hs = agg(P, "final_heldout_margin")
    pm, ps = agg(P, "final_para_margin")
    pp, _  = agg(P, "final_ppl")
    print(f"{P:<12} {tm:>+8.2f}±{ts:<4.2f} {hm:>+8.2f}±{hs:<4.2f} "
          f"{pm:>+8.2f}±{ps:<4.2f} {pp:>10.1f}")

g_tr, _ = agg("GATE_ONLY", "final_train_margin")
b_tr, _ = agg("BEATRIZ",   "final_train_margin")
g_ho, _ = agg("GATE_ONLY", "final_heldout_margin")
b_ho, _ = agg("BEATRIZ",   "final_heldout_margin")
print("\n>>> APORTE AISLADO DE LA PERDIDA CONTRASTIVA (BEATRIZ - GATE_ONLY):")
print(f"    Train margin  : {b_tr - g_tr:+.2f}")
print(f"    Held-out margin: {b_ho - g_ho:+.2f}")

gate_ms, _ = agg("BEATRIZ", "gate_ms_per_call")
vram, _    = agg("BEATRIZ", "peak_vram_gb")
print(f"\n>>> COSTO: gate {gate_ms:.3f} ms/llamada | VRAM pico {vram:.2f} GB "
      f"| total {total_time/60:.1f} min ({len(SEEDS)*len(POLICIES)} corridas)")

report = {
    "metadata": {
        "experiment": EXPERIMENT, "author": AUTHOR, "year": YEAR, "license": LICENSE,
        "base_model": BASE_MODEL_NAME, "lora_targets": LORA_TARGET_MODULES,
        "policies": POLICIES, "seeds": SEEDS, "epochs": EPOCHS,
        "draws_per_epoch": DRAWS_PER_EPOCH, "p_lie_schedule": P_LIE_SCHEDULE,
        "alpha": ALPHA, "beta": BETA, "margin": MARGIN, "lr": LR, "fp16": USE_FP16,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "train_size": len(TRAIN_BENCHMARK), "heldout_size": len(HELDOUT_BENCHMARK),
        "ppl_corpus_size": len(NEUTRAL_EVAL_TEXTS),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "oracle_cache_time_s": oracle_time,
        "total_wallclock_s": total_time,
    },
    "results": results
}
with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*80)
print("EXP15 TERMINADO")
print(f"Reporte: {FINAL_FILE}")
print(f"SHA-256: {sha256_file(FINAL_FILE)}")
print("="*80)


EXP15 — Beatriz Ablation Test (none / gate_only / beatriz)
Author: Eduardo Ayala Tovar | Year: 2026 | License: PolyForm Noncommercial License 1.0.0
Device: cuda | Base: microsoft/Phi-3-mini-4k-instruct | Targets: ['qkv_proj', 'o_proj']
GPU: Tesla T4


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  [BASE] Train +1.34 | Held-Out +1.90 | Para +2.21 | PPL 12.7 | Var 0.5932
  Ep 1/8 | Loss 0.7416 | Train +0.83 | Held-Out +2.30 | Para +2.16 | contrast 0
  Ep 2/8 | Loss 0.2556 | Train -0.02 | Held-Out +3.03 | Para +2.96 | contrast 0
  Ep 3/8 | Loss 0.0679 | Train -0.09 | Held-Out +3.47 | Para +3.04 | contrast 0
  Ep 4/8 | Loss 0.0356 | Train -0.01 | Held-Out +3.47 | Para +2.86 | contrast 0
  Ep 5/8 | Loss 0.0619 | Train -0.03 | Held-Out +3.49 | Para +3.06 | contrast 0
  Ep 6/8 | Loss 0.0241 | Train -0.06 | Held-Out +3.60 | Para +3.70 | contrast 0
  Ep 7/8 | Loss 0.0336 | Train -0.07 | Held-Out +3.64 | Para +3.41 | contrast 0
  Ep 8/8 | Loss 0.0335 | Train -0.03 | Held-Out +3.80 | Para +3.47 | contrast 0
  [FINAL] PPL 12.7->31.8 | Held-Out +3.80 (best ep8 +3.80) | Var 0.5932->1.0800 | gate 0.008 ms/call | 86.8s | 7.86 GB

[RAMA GATE_ONLY] Semilla 33


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

  [BASE] Train +1.34 | Held-Out +1.90 | Para +2.21 | PPL 12.7 | Var 0.5932
  Ep 1/8 | Loss 0.4523 | Train +5.98 | Held-Out +4.70 | Para +3.86 | contrast 0
  Ep 2/8 | Loss 0.0274 | Train +7.57 | Held-Out +5.17 | Para +4.47 | contrast 0
  Ep 3/8 | Loss 0.0002 | Train +7.59 | Held-Out +5.20 | Para +4.41 | contrast 0
  Ep 4/8 | Loss 0.0001 | Train +7.60 | Held-Out +5.20 | Para +4.40 | contrast 0
  Ep 5/8 | Loss 0.0000 | Train +7.61 | Held-Out +5.21 | Para +4.39 | contrast 0
  Ep 6/8 | Loss 0.0000 | Train +7.62 | Held-Out +5.20 | Para +4.38 | contrast 0
  Ep 7/8 | Loss 0.0000 | Train +7.62 | Held-Out +5.21 | Para +4.39 | contrast 0
  Ep 8/8 | Loss 0.0000 | Train +7.62 | Held-Out +5.21 | Para +4.38 | contrast 0
  [FINAL] PPL 12.7->66.0 | Held-Out +5.21 (best ep8 +5.21) | Var 0.5932->1.5513 | gate 0.104 ms/call | 72.1s | 7.86 GB

[RAMA BEATRIZ] Semilla 33


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

  [BASE] Train +1.34 | Held-Out +1.90 | Para +2.21 | PPL 12.7 | Var 0.5932
  Ep 1/8 | Loss 0.4809 | Train +7.14 | Held-Out +5.37 | Para +4.21 | contrast 13
  Ep 2/8 | Loss 0.0370 | Train +8.60 | Held-Out +5.72 | Para +4.80 | contrast 25
  Ep 3/8 | Loss 0.0003 | Train +9.11 | Held-Out +5.83 | Para +4.84 | contrast 23
  Ep 4/8 | Loss 0.0002 | Train +9.60 | Held-Out +5.86 | Para +4.89 | contrast 25
  Ep 5/8 | Loss 0.0001 | Train +9.86 | Held-Out +5.86 | Para +4.90 | contrast 25
  Ep 6/8 | Loss 0.0001 | Train +10.03 | Held-Out +5.87 | Para +4.92 | contrast 28
  Ep 7/8 | Loss 0.0001 | Train +10.13 | Held-Out +5.88 | Para +4.94 | contrast 32
  Ep 8/8 | Loss 0.0001 | Train +10.22 | Held-Out +5.89 | Para +4.94 | contrast 33
  [FINAL] PPL 12.7->87.4 | Held-Out +5.89 (best ep8 +5.89) | Var 0.5932->1.6908 | gate 0.105 ms/call | 103.3s | 7.97 GB

RESUMEN DE ABLACION (media ± std, n=3)
Rama                  Train       Held-Out           Para        PPL
BASE                  +1.34          +1.90   